# Data Cleaning
This notebook is used to apply the cleaning steps to the raw dataset and explain the reasons behind each transformation/decision.

In [1]:
import pandas as pd

df = pd.read_csv("../data/naloxone.csv")

## Renaming Columns to snake_case
The original column names use title case with spaces, which makes them a bit harder to work with in Python. When it comes to coding I prefer to use snake_case naming convention. Renaming everything to snake_case is a small change that keeps the rest of the code cleaner. In this step I also fixed a typo in `Naxolone Administrations`. The correct spelling is *Na**lo**xone*.

In [2]:
df = df.rename(columns=lambda x: x.replace(" ", "_").lower())

In [3]:
df = df.rename(columns={
    "naxolone_administrations": "naloxone administrations"
})

## Dropping Unecessary Columns
Three columns were dropped because they don't add much analytical value to the analysis:
- `id`: just a composite key made from `incident_number` and `patient_number`.

- `neighbourhood`: has 223 unique values, many with only one incident, which makes it hard to analyze.
    - `ward` already does a great job covering this grouped geographical information into 15 categories.

- `neighbourhood_id`: just a code for each neighbourhood we already dropped. 

In [4]:
df = df.drop(columns=[
    "id",
    "neighbourhood",
    "neighbourhood_id"
])

## Parsing `dispatch_date`
`dispatch_date` was stored as a plain string (`"2021-08-08T04:33:22"`), which means pandas treats it as text. The best way to filter / extract information from this column is converting it to real datetime data type. After parsing it to datetime, I extracted `year`, `month`, `day_of_week`, and `hour` as separate columns. These are temporal dimensions columns that I plan to use to groupby/filter in the analysis. I also kept a `date` column (date only, no time) as a reference. The original `dispatch_date` was dropped after extracting the other columns out since the needed information lives in those new columns.

In [5]:
# extracting temporal components of "dispatch_date"


# parsing column to datetime
df["dispatch_date"] = pd.to_datetime(df["dispatch_date"])

# creating "date" column to only hold the date
df["date"] = df["dispatch_date"].dt.date

# creating "year" column to only hold the year
df["year"] = df["dispatch_date"].dt.year

# creating "month" column to only hold the month
df["month"] = df["dispatch_date"].dt.month

# creating "day_of_week" column to hold only the day of the week
df["day_of_week"] = df["dispatch_date"].dt.day_of_week

# creating "hour" column to only hold the hour
df["hour"] = df["dispatch_date"].dt.hour

# dropping now redundant column, "dispatch_date"
df = df.drop(columns=["dispatch_date"])